# Calculations and demos of the particle and field handling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### White noise generation

In [ ]:
Lbox = [1000, 400, 200]
x = np.random.random((1000, 3)) * np.array(Lbox)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    ax.scatter(*x[:, idx].T, color='red', s=4**2, ec='none', alpha=0.5)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')

plt.show()

In [ ]:
def cubic_voxels(nmesh, Lbox, periodic, silent=False):
    '''
    Defines a rectangular cuboid mesh with the specified number of
    voxels in each dimensions, ensuring that the voxels are cubic.
    The function calculates the number of voxels in each dimension
    (Nx, Ny, Nz) based on the shortest dimension of the cuboid and
    scales the other dimensions accordingly.

    Parameters:
    -----------
    nmesh : int
        Number of voxels in the shortest dimension.
    Lbox : float or list of float
        Length of the box in each dimension [Lx, Ly, Lz].
    periodic : bool or list of bool
        If True, the box is periodic in all dimensions. Arbitrary
        combinations of periodic and open boundary conditions can be
        specified for each dimension as a list of three booleans.
    silent : bool
        If True, suppresses output messages.
    '''
    if not isinstance(Lbox, (list, tuple)):
        Lbox = (Lbox,) * 3
    if not isinstance(periodic, (list, tuple)):
        periodic = (periodic,) * 3
    ref_L = np.min(Lbox)
    mesh = np.ceil(Lbox / (ref_L / nmesh)).astype(int)
    mesh = (mesh + mesh % 2).astype(int)  # Ensure even number of voxels
    mesh += 1 - np.array(periodic, dtype=int)  # Add 1 if open boundary condition
    if not silent:
        print('Mesh: Nx={}, Ny={}, Nz={}'.format(*mesh))
    dk = ref_L / mesh[Lbox.index(ref_L)]
    if not silent:
        print(f'Step size: {dk}')
    return dk, mesh

In [ ]:
nmesh = 8
Lbox = [100, 100, 50]
periodic = [0, 0, 1]
_, nvox = cubic_voxels(nmesh, Lbox, periodic)

mesh = [np.linspace(0, L, m, endpoint=1-p) for L, m, p in zip(Lbox, nvox, periodic)]
grid = np.meshgrid(*mesh, indexing='ij')
field = np.random.normal(size=nvox)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    
    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]
    
    ax.scatter(x1_s, x2_s, color='red', s=6**2, ec='none', alpha=0.5)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)

    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]
    
    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.scatter(x1_s, x2_s, c=field_s, s=10**2, ec='none', alpha=0.5)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)

    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.pcolormesh(x1_s, x2_s, field_s, shading='auto', cmap='viridis')
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title(f'{labels[idx[0]]}{labels[idx[1]]} slice at {labels[i]}={c_i[i]}')
plt.show()

### Field interpolation to particle positions

In [ ]:
from scipy.interpolate import RegularGridInterpolator

In [ ]:
def interpolate_field(x, field, Lbox, periodic, method='linear'):
    r'''
    Interpolate a grid-based field onto particle positions using periodic
    boundaries.

    Parameters
    ----------
    x : ndarray of shape (N, 3)
        Particle positions in the simulation box.
    field : ndarray
        The grid-based field (e.g., a displacement field) defined on a
        regular grid.
    periodic : bool or list of bool
        If True, the box is periodic in all dimensions. Arbitrary
        combinations of periodic and open boundary conditions can be
        specified for each dimension as a list of three booleans.
    Lbox : float
        The simulation box size.
    method : str, optional
        The interpolation method to use. This can be 'linear', 'nearest',
        or 'cubic'. The default is 'linear'.

    Returns
    -------
    interp_values : ndarray of shape (N,)
        Field values interpolated at the particle positions.
    '''
    if not isinstance(Lbox, (list, tuple)):
        Lbox = (Lbox,) * 3
    if not isinstance(periodic, (list, tuple)):
        periodic = (periodic,) * 3
    nvox = field.shape
    mesh = [np.linspace(0, L, n, endpoint=1-p) for L, n, p in zip(Lbox, nvox, periodic)]
    interpolator = RegularGridInterpolator(
        tuple(mesh),
        field,
        method=method,
        bounds_error=False,
        fill_value=None  # Extrapolate using periodic wrapping if needed
    )
    return interpolator(np.mod(x, Lbox))

In [ ]:
nmesh = 32
Lbox = [100, 100, 50]
periodic = [0, 0, 1]
_, nvox = cubic_voxels(nmesh, Lbox, periodic)

field = np.random.normal(size=nvox)
mesh = [np.linspace(0, L, m, endpoint=1-p) for L, m, p in zip(Lbox, nvox, periodic)]
grid = np.meshgrid(*mesh, indexing='ij')

x = np.random.random((2000, 3)) * np.array(Lbox)
field_interp = interpolate_field(x, field, Lbox, periodic)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)

    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]
    
    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.scatter(x1_s, x2_s, c=field_s, s=4**2, ec='none', alpha=1.0)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title(f'{labels[idx[0]]}{labels[idx[1]]} slice at {labels[i]}={c_i[i]}')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    
    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.pcolormesh(x1_s, x2_s, field_s, shading='auto', cmap='viridis')
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title(f'{labels[idx[0]]}{labels[idx[1]]} slice at {labels[i]}={c_i[i]}')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    ax.scatter(*x[:, idx].T, c=field_interp, s=4**2, ec='none', alpha=0.5)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()